In [5]:
import os 
import numpy as np
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from PIL import Image 
import matplotlib.pyplot as plt
import shutil
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt
import os
import random
import shutil


In [ ]:
np.version.version

In [6]:
data_dir = "/Users/adi/Classification/CUB_200_2011/CUB_200_2011/images/"

In [ ]:
species_list = os.listdir(data_dir)
print("Classes:", species_list)
print("shape: {0}".format(len(species_list)))


In [ ]:
from tensorflow.keras.applications import InceptionV3
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import GlobalAveragePooling2D, Dense, Dropout
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.regularizers import l2


# Load InceptionV3 model pre-trained on ImageNet
base_model = InceptionV3(weights='imagenet', include_top=False, input_shape=(224, 224, 3))

# Freeze the layers of InceptionV3
for layer in base_model.layers:
    layer.trainable = False

# Build the model
model = Sequential()
model.add(base_model)
model.add(GlobalAveragePooling2D())
model.add(Dense(128, activation='relu', kernel_regularizer=l2(0.01)))  # Apply L2 regularization
model.add(Dropout(0.6))
model.add(Dense(len(species_list), activation='softmax'))

# Compile the model
model.compile(optimizer=Adam(learning_rate=1e-4), loss='categorical_crossentropy', metrics=['accuracy'])

# Display the model summary
model.summary()

# Data augmentation for training
train_datagen = ImageDataGenerator(
    rescale=1./255,                # Rescale pixel values
    rotation_range=20,             # Random rotations
    width_shift_range=0.2,         # Horizontal shifts
    height_shift_range=0.2,        # Vertical shifts
    shear_range=0.2,               # Shearing
    zoom_range=0.2,                # Zooming
    horizontal_flip=True,          # Horizontal flipping
    fill_mode='nearest'            # Fill empty pixels
)

# Test data should only be rescaled
test_datagen = ImageDataGenerator(rescale=1./255)

# Data generators
train_generator = train_datagen.flow_from_directory(
    "/Users/adi/Classification/bird_train",  # Dataset path
    target_size=(224, 224),
    batch_size=32,
    class_mode='categorical'
)

test_generator = test_datagen.flow_from_directory(
    "/Users/adi/Classification/bird_test",  # Dataset path
    target_size=(224, 224),
    batch_size=32,
    class_mode='categorical'
)

# Callbacks
early_stopping = EarlyStopping(
    monitor='val_loss',
    patience=5,
    restore_best_weights=True
)

reduce_lr = ReduceLROnPlateau(
    monitor='val_loss',
    factor=0.2,
    patience=3,
    min_lr=1e-6
)

# Train the model
history = model.fit(
    train_generator,
    epochs=50,
    validation_data=test_generator,
    callbacks=[early_stopping, reduce_lr]
)

# Fine-tuning: Unfreeze some layers of the base model
for layer in base_model.layers[-30:]:  # Unfreeze the last 30 layers
    layer.trainable = True

# Recompile the model with a lower learning rate for fine-tuning
model.compile(optimizer=Adam(learning_rate=1e-5), loss='categorical_crossentropy', metrics=['accuracy'])

# Fine-tune the model
fine_tune_history = model.fit(
    train_generator,
    epochs=30,
    validation_data=test_generator,
    callbacks=[early_stopping, reduce_lr]
)

# Save the model
model.save("fine_tuned_inceptionv3.h5")


In [ ]:
# Plot training and Validation accuracy 

plt.figure(figsize=(12,4))
plt.subplot(1,2,1)
plt.plot(history.history['accuracy'], label='Training Accuracy')
plt.plot(history.history['val_accuracy'], label='Test Accuracy')
plt.title('Model Accuracy')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.legend()

#Plot training and validation loss

plt.subplot(1,2,2)
plt.plot(history.history['loss'], label='Training Loss')
plt.plot(history.history['val_loss'], label='Test Loss')
plt.title('Model Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()

plt.show()


In [ ]:
 #Evaluate Model on validation set 

val_loss, val_accuracy = model.evaluate(test_data)
print(f"Validation Accuracy: {val_accuracy * 100:.2f}")
print(f"Validation Loss: {val_loss:.4f}")# 